# Filterdiagnostik für Transaktionsdaten

Dieses Notebook vergleicht die ungefilterten Transaktionen in `data/interim/transactions_per_year` mit dem gefilterten Output in `data/interim/transactions_per_year_filtered` (<- hier noch separate Transaktionen).

Es zeigt die Anzahl der Reihen, Produkte und Filialen, den Effekt der einzelnen Filter und die aktivsten Produkte, Filialen, Mandanten und Warengruppen im gefilterten Datensatz.


## Setup

Die Filterbedingungen werden direkt aus `src.data.cleaning.rules` importiert, damit die Auswertung dieselbe Logik wie die Pipeline verwendet.


In [1]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.common import read_parquet_expr
from src.data.cleaning.rules import (
    ALLOWED_FCM_ARTICLE_IDS,
    ALLOWED_MANDANT_IDS,
    FCM_RULE,
    MANDANT_RULE,
    MIN_UMS_MENGE,
    WEIGHT_CONTENT_LIKE,
    WEIGHT_RULE,
    fcm_filter_condition,
    mandant_filter_condition,
    transaction_filter_condition,
    ums_menge_filter_condition,
    weight_filter_condition,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

IN_DIR = ROOT / "data" / "interim" / "transactions_per_year"
OUT_DIR = ROOT / "data" / "interim" / "transactions_per_year_filtered"
IN_GLOB = IN_DIR / "transactions_year_*.parquet"
OUT_GLOB = OUT_DIR / "transactions_year_*.parquet"

if not list(IN_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {IN_DIR} gefunden")
if not list(OUT_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {OUT_DIR} gefunden")

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("SET preserve_insertion_order=false")

raw_expr = read_parquet_expr(IN_GLOB)
filtered_expr = read_parquet_expr(OUT_GLOB)

print(f"IN_DIR:  {IN_DIR}")
print(f"OUT_DIR: {OUT_DIR}")


IN_DIR:  /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year
OUT_DIR: /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year_filtered


## Filterstatus und technische Bedingungen

Diese Tabelle zeigt, welche Filter aktiv sind und welche konkrete SQL-Bedingung aus der Pipeline verwendet wird. Die fachliche Bedeutung der Filter steht separat unter der Tabelle.


In [2]:
filter_definitions = pd.DataFrame([
    {
        "Filter": "UMS_MENGE",
        "aktiv": True,
        "Bedingung": ums_menge_filter_condition(),
        "Parameter": f"MIN_UMS_MENGE = {MIN_UMS_MENGE}",
    },
    {
        "Filter": "MANDANT_ID",
        "aktiv": MANDANT_RULE,
        "Bedingung": mandant_filter_condition() if MANDANT_RULE else "deaktiviert",
        "Parameter": f"erlaubte Mandanten = {sorted(ALLOWED_MANDANT_IDS)}" if MANDANT_RULE else "-",
    },
    {
        "Filter": "FCM-Artikel",
        "aktiv": FCM_RULE,
        "Bedingung": fcm_filter_condition() if FCM_RULE else "deaktiviert",
        "Parameter": f"{len(ALLOWED_FCM_ARTICLE_IDS):,} erlaubte Artikel-IDs" if FCM_RULE else "-",
    },
    {
        "Filter": "ARTIKEL_INHALT Gewicht",
        "aktiv": WEIGHT_RULE,
        "Bedingung": weight_filter_condition() if WEIGHT_RULE else "deaktiviert",
        "Parameter": f"ARTIKEL_INHALT LIKE {WEIGHT_CONTENT_LIKE!r}" if WEIGHT_RULE else "-",
    },
])

filter_definitions


,Filter,aktiv,Bedingung,Parameter
0,UMS_MENGE,True,"""UMS_MENGE"" > 0.005",MIN_UMS_MENGE = 0.005
1,MANDANT_ID,True,"""MANDANT_ID"" IN (110, 130, 135)","erlaubte Mandanten = [110, 130, 135]"
2,FCM-Artikel,True,"""ARTIKEL_ID"" IN (560039, 579781, 1376569, 1376...",103 erlaubte Artikel-IDs
3,ARTIKEL_INHALT Gewicht,True,"(CASE WHEN LOWER(COALESCE(CAST(""ARTIKEL_INHALT...",ARTIKEL_INHALT LIKE '%amm%'


## Fachliche Bedeutung der Filter

- `UMS_MENGE`: Behält nur Transaktionszeilen mit einer relevanten positiven Verkaufsmenge. Dadurch werden Nullmengen, Kleinstmengen unterhalb des Schwellwerts und negative Mengen entfernt.
- `MANDANT_ID`: Beschränkt die Daten auf die für die Analyse vorgesehenen Mandanten.
- `FCM-Artikel`: Beschränkt die Daten auf die definierte FCM-Artikelliste und damit auf die untersuchte Produktauswahl.
- `ARTIKEL_INHALT Gewicht`: Behält Produkte, bei denen eine Abverkaufsmenge in kg ableitbar ist. Die Filterregel prüft dafür die Inhaltsangabe `ARTIKEL_INHALT` auf Gramm-/Kilogramm-Angaben. Im gefilterten Output wird `ABVERKAUFTE_MENGE_KG` anschließend aus `GRAMM_BON` gebildet: Für echte Gewichtsartikel (`GEWICHTSARTIKEL = 1`) wird `GRAMM_BON` direkt verwendet; bei verpackten Artikeln wird `GRAMM_BON` nur verwendet, wenn `ARTIKEL_INHALT` eine Gewichtseinheit wie `kg`, `Kilogramm`, `g`, `gr.` oder `Gramm` enthält. Für andere Artikel wäre keine belastbare kg-Menge ableitbar.


## Umfang vor und nach dem Filtern

Eine Reihe ist hier eine eindeutige Kombination aus `ARTIKEL_ID` und `MARKT_ID`.


In [3]:
def where_clause(condition):
    return f"WHERE {condition}" if condition else ""


def count_snapshot(expr, condition=None):
    where_sql = where_clause(condition)
    return con.execute(
        f"""
        SELECT
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(*) FILTER (WHERE ARTIKEL_ID IS NOT NULL AND MARKT_ID IS NOT NULL)::BIGINT AS Zeilen_mit_Reihenschluessel,
            COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))::BIGINT AS Reihen,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            MIN(CAST(DATE AS DATE)) AS erster_Tag,
            MAX(CAST(DATE AS DATE)) AS letzter_Tag
        FROM {expr}
        {where_sql}
        """
    ).fetchdf().iloc[0].to_dict()


overview = pd.DataFrame(
    [
        {"Datensatz": "ungefiltert", **count_snapshot(raw_expr)},
        {"Datensatz": "gefiltert", **count_snapshot(filtered_expr)},
    ]
)
overview["entfernte_Zeilen"] = overview["Zeilen"].iloc[0] - overview["Zeilen"]
overview["behaltener_Zeilenanteil_%"] = (100 * overview["Zeilen"] / overview["Zeilen"].iloc[0]).round(2)

overview


,Datensatz,Zeilen,Zeilen_mit_Reihenschluessel,Reihen,Produkte,Filialen,erster_Tag,letzter_Tag,entfernte_Zeilen,behaltener_Zeilenanteil_%
0,ungefiltert,13495921,13495921,24973,4843,43,2021-07-01,2026-06-30,0,100.00
1,gefiltert,41506,41506,298,43,21,2025-04-19,2026-06-30,13454415,0.31


## Effekt der einzelnen Filter

Die Filter werden sequenziell ausgewertet. `entfernt_*` bedeutet daher: zusätzlich entfernt nach allen vorherigen Filtern.


In [4]:
def combine_conditions(left, right):
    return f"({left}) AND ({right})" if left else right


filter_steps = [("UMS_MENGE", ums_menge_filter_condition())]
if MANDANT_RULE:
    filter_steps.append(("MANDANT_ID", mandant_filter_condition()))
if FCM_RULE:
    filter_steps.append(("FCM-Artikel", fcm_filter_condition()))
if WEIGHT_RULE:
    filter_steps.append(("ARTIKEL_INHALT Gewicht", weight_filter_condition()))

impact_rows = []
current_condition = None
previous = count_snapshot(raw_expr)

for filter_name, filter_condition in filter_steps:
    current_condition = combine_conditions(current_condition, filter_condition)
    current = count_snapshot(raw_expr, current_condition)
    row = {
        "Filter": filter_name,
        "vorher_Zeilen": previous["Zeilen"],
        "nachher_Zeilen": current["Zeilen"],
        "entfernt_Zeilen": previous["Zeilen"] - current["Zeilen"],
        "entfernt_Zeilen_%": round(
            100 * (previous["Zeilen"] - current["Zeilen"]) / previous["Zeilen"], 2
        ) if previous["Zeilen"] else 0,
        "vorher_Reihen": previous["Reihen"],
        "nachher_Reihen": current["Reihen"],
        "entfernt_Reihen": previous["Reihen"] - current["Reihen"],
        "vorher_Produkte": previous["Produkte"],
        "nachher_Produkte": current["Produkte"],
        "entfernt_Produkte": previous["Produkte"] - current["Produkte"],
        "vorher_Filialen": previous["Filialen"],
        "nachher_Filialen": current["Filialen"],
        "entfernt_Filialen": previous["Filialen"] - current["Filialen"],
    }
    impact_rows.append(row)
    previous = current

filter_impact = pd.DataFrame(impact_rows)
expected_kept = count_snapshot(raw_expr, transaction_filter_condition())
actual_kept = count_snapshot(filtered_expr)

if expected_kept["Zeilen"] != actual_kept["Zeilen"]:
    print(
        "WARNUNG: Die erwartete Zeilenzahl nach Filterlogik weicht vom vorhandenen OUT_DIR ab: "
        f"erwartet={expected_kept['Zeilen']:,}, OUT_DIR={actual_kept['Zeilen']:,}"
    )

filter_impact


,Filter,vorher_Zeilen,nachher_Zeilen,entfernt_Zeilen,entfernt_Zeilen_%,vorher_Reihen,nachher_Reihen,entfernt_Reihen,vorher_Produkte,nachher_Produkte,entfernt_Produkte,vorher_Filialen,nachher_Filialen,entfernt_Filialen
0,UMS_MENGE,13495921,13495727,194,0.00,24973,24973,0,4843,4843,0,43,43,0
1,MANDANT_ID,13495727,7991131,5504596,40.79,24973,12582,12391,4843,2245,2598,43,21,22
2,FCM-Artikel,7991131,41506,7949625,99.48,12582,298,12284,2245,43,2202,21,21,0
3,ARTIKEL_INHALT Gewicht,41506,41506,0,0.00,298,298,0,43,43,0,21,21,0


## Aktivste Entitäten im gefilterten Datensatz

Aktivität wird für Produkte, Filialen, Mandanten und Warengruppen über die Anzahl der Transaktionszeilen gemessen. Ergänzend werden Tage, Produkte, Filialen, Menge und Umsatz ausgewiesen, soweit sie für die jeweilige Entität sinnvoll sind.


In [5]:
def top_entities(group_cols, label_cols=None, top_n=10, order_by="Zeilen DESC"):
    label_cols = label_cols or []
    select_cols = []
    group_sql = []
    for col in group_cols:
        select_cols.append(col)
        group_sql.append(col)
    for col in label_cols:
        select_cols.append(f"arg_max({col}, DATE) AS {col}")

    select_sql = ",\n            ".join(select_cols)
    group_by_sql = ", ".join(group_sql)

    return con.execute(
        f"""
        SELECT
            {select_sql},
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(DISTINCT CAST(DATE AS DATE))::BIGINT AS Nachfragetage,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            SUM(COALESCE(UMS_MENGE, 0.0))::DOUBLE AS Summe_UMS_MENGE,
            SUM(COALESCE(UMS_VK_WERT, 0.0))::DOUBLE AS Summe_UMS_VK_WERT
        FROM {filtered_expr}
        GROUP BY {group_by_sql}
        ORDER BY {order_by}
        LIMIT {top_n}
        """
    ).fetchdf()


def display_top(title, dataframe):
    print(title)
    display(dataframe)


In [6]:
top_products = top_entities(
    group_cols=["ARTIKEL_ID"],
    label_cols=["ARTIKEL_BEZ", "ARTIKEL_INHALT", "VERKAUFSEINHEIT", "GEWICHTSARTIKEL", "WGR_ID", "N_WARENKLASSE_KBEZ"],
    order_by="Zeilen DESC",
)
display_top("Aktivste Produkte nach Transaktionszeilen", top_products)


Aktivste Produkte nach Transaktionszeilen


,ARTIKEL_ID,ARTIKEL_BEZ,ARTIKEL_INHALT,VERKAUFSEINHEIT,GEWICHTSARTIKEL,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,1382764,Braunschweiger grobe Streichmettwurst,125 Gramm,St,0,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",5795,322,1,10,6631.0000,10123.9201
1,1382765,Braunschweiger feine Streichmettwurst,125 Gramm,St,0,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",5580,322,1,9,6712.0000,10739.3699
2,1382777,Goldmarie gekochte Schweinemettwurst,1 Kilogramm,kg,1,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",5497,286,1,7,550.4680,7812.6500
3,1382767,Fleischwurst ohne Knoblauch,1 Kilogramm,kg,1,900,Fleischwurst,4402,349,1,9,1130.2170,15148.7300
4,1382768,Fleischkäse ofengebacken,1 Kilogramm,kg,1,900,Fleischkäseprodukte,3255,350,1,8,999.1210,11265.3099
5,1382766,Fleischwurst mit Knoblauch,1 Kilogramm,kg,1,900,Fleischwurst,2947,349,1,9,736.8670,9907.1000
6,1406063,SB Fleischkäse roh in Schale,1 Kilogramm,kg,1,900,Fleischkäseprodukte,1740,178,1,9,837.2634,5666.4100
7,1376860,Pfefferbeißer,1 Kilogramm,kg,1,900,"Rohwurst, Stückware",1559,262,1,6,293.6100,5466.3200
8,1413121,Schweinebraten-Aufschnitt Spießbraten Art,1 Kilogramm,kg,1,900,"Bratenaufschnitt, Rind-, Schwein",1312,131,1,10,156.5510,2649.6500
9,1382788,"Goldmarie Friesen Bratwurst, gebrüht",1 Kilogramm,kg,1,900,Bratwurst,1118,176,1,6,1143.4020,4945.0804


In [7]:
top_stores = top_entities(
    group_cols=["MARKT_ID"],
    label_cols=["MARKT_NR", "MANDANT_ID", "LEH_SEH"],
)
display_top("Aktivste Filialen", top_stores)


Aktivste Filialen


,MARKT_ID,MARKT_NR,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,1300023,23,130,LEH,7788,341,25,1,3716.3360,20697.7400
1,1100045,45,110,LEH,3303,332,18,1,2657.4766,8716.0605
2,1350061,61,135,SEH,3271,314,13,1,1094.1050,9294.5700
3,1100055,55,110,LEH,2949,344,30,1,1743.0475,9312.0400
4,1300021,21,130,LEH,2796,322,9,1,1973.7866,7107.8700
5,1100076,76,110,LEH,2534,347,28,1,1461.8538,8442.6799
6,1100059,59,110,LEH,2175,315,27,1,1618.3907,7706.1900
7,1300024,24,130,LEH,1863,309,7,1,1625.1245,4532.6000
8,1100039,39,110,LEH,1801,316,19,1,448.4814,5194.4900
9,1100051,51,110,LEH,1799,328,20,1,324.1551,4723.4100


In [8]:
top_mandants = top_entities(
    group_cols=["MANDANT_ID"],
    label_cols=["LEH_SEH"],
)
display_top("Aktivste Mandanten", top_mandants)


Aktivste Mandanten


,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,110,LEH,23142,360,43,15,12765.0182,69220.0303
1,130,LEH,15093,351,35,5,8546.2993,39932.3699
2,135,SEH,3271,314,13,1,1094.1050,9294.5700


In [9]:
top_warengruppen = top_entities(
    group_cols=["WGR_ID"],
    label_cols=["N_WARENKLASSE_KBEZ"],
)
display_top("Aktivste Warengruppen", top_warengruppen)


Aktivste Warengruppen


,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,900,Fleischwurst,39617,361,26,20,20519.8310,100509.2803
1,890,Bacon,1889,218,17,19,1885.5915,17937.6899


## Hinweise

Die Filterwirkung ist sequenziell berechnet. Wenn ein Datensatz bereits durch einen früheren Filter entfernt wurde, wird er bei späteren Filtern nicht erneut gezählt.
